# 🌶️ PEDAS 2026 — Notebook v4

Pipeline lengkap: **audit mislabel → feature engineering → training → prediksi.**

---

## 1. Import Library

In [1]:
import os
import sys
sys.path.insert(0, '../')  # supaya bisa import audit_mislabel.py dari folder app/

import pandas as pd
from audit_mislabel import audit_mislabel, apply_relabel, summarize_audit, summarize_diff

pd.set_option('display.max_columns', None)

## 2. Ambil Data

In [2]:
train = pd.read_csv('../../data/raw/training.csv')
test  = pd.read_csv('../../data/raw/predict.csv')

print(f"Dimensi Training : {train.shape}")
print(f"Dimensi Predict  : {test.shape}")
train.head(3)

Dimensi Training : (8400, 10)
Dimensi Predict  : (1500, 10)


,url,brand,discovered,confidence_level,ip,domain,sld,category,registrar,registration_date
0,https://lucah3.***********.my.id/,Telegram,5/1/2024 21:52,100,NaN,***********.my.id,my.id,phishingg,PT Web Media Technology Indonesia,5/14/2026
1,https://dpmptsp.*********.go.id/petapotensi/da...,judi online,8/5/2024 20:41,100,103.162.68.84,*********.go.id,go.id,online gambling,Kementerian Komunikasi dan Informatika,5/11/2009
2,http://klikpad.bkpd.**************.go.id/klikp...,-,6/14/2024 7:05,100,103.18.117.8,**************.go.id,go.id,online gambling,Kementerian Komunikasi dan Informatika,3/13/2008


## 3. Bersihkan Penulisan Category

Normalisasi typo & inkonsistensi kapital pada kolom `category` — **hanya diterapkan ke data train**.

In [3]:
cat_map = {
    'online gamblingg': 'online gambling', 'Online Gambling': 'online gambling',
    'phishingg': 'phishing', 'otherr': 'other', 'Other': 'other',
    'malwaree': 'malware', 'spamm': 'spam', 'Brand': 'brand',
    'FakeShop': 'fakeshop', 'PIIExposure': 'pii_exposure',
}
train['category'] = train['category'].str.strip().replace(cat_map).str.lower().str.strip()

print("=== Distribusi Kategori Setelah Pembersihan ===")
print(train['category'].value_counts())

=== Distribusi Kategori Setelah Pembersihan ===
category
online gambling    5447
phishing           2253
other               284
spam                185
malware             179
brand                45
fakeshop              5
violence              1
pii_exposure          1
Name: count, dtype: int64


## 4. Audit & Perbaiki Category

Mendeteksi potensi mislabel menggunakan bukti dari kolom `url` + `ip` — **hanya train**.

In [4]:
train_audited = audit_mislabel(train)
summarize_audit(train_audited)

=== Distribusi skor kecurigaan ===
suspect_score
11       1
8       24
7        1
6        1
5       34
4       10
3      129
2      134
1      132
0     7934
Name: count, dtype: int64

Skor >=5 (relabel otomatis)      : 61 baris
Skor 3-4 (verifikasi manual)     : 139 baris
Skor 1-2 (biarkan, terlalu lemah): 266 baris


## 5. Terapkan Relabel Category

Relabel diterapkan jika **skor audit ≥ 5**. Diff disimpan ke `data/result/v4/relabel_diff.csv`.

In [5]:
train, diff_table = apply_relabel(train_audited, score_threshold=5)
summarize_diff(diff_table)

os.makedirs('../../data/result/v4', exist_ok=True)
diff_table.to_csv('../../data/result/v4/relabel_diff.csv', index=False)

=== Total baris yang benar-benar berubah label: 61 ===

=== Perubahan per pasangan (label lama -> label baru) ===
category_before  category_after 
phishing         online gambling    20
malware          online gambling    14
spam             online gambling    12
other            online gambling     8
online gambling  phishing            6
spam             phishing            1
dtype: int64


## 6. Gabung Kategori Super Langka

Kategori dengan jumlah baris **< 5** digabungkan ke kelas `other` untuk menghindari masalah saat stratified split.

In [6]:
category_counts = train['category'].value_counts()
rare_categories = category_counts[category_counts < 5].index.tolist()
print("Kategori digabung ke 'other':", rare_categories)

train['category'] = train['category'].replace({cat: 'other' for cat in rare_categories})
print("\nDistribusi category final:")
print(train['category'].value_counts())

Kategori digabung ke 'other': ['violence', 'pii_exposure']

Distribusi category final:
category
online gambling    5495
phishing           2240
other               278
spam                172
malware             165
brand                45
fakeshop              5
Name: count, dtype: int64


## 7. Parse Format Tanggal

Konversi kolom tanggal ke tipe `datetime` — diterapkan ke **train & test**.

> `discovered_hour` / `dayofweek` **tidak dipakai** — terbukti artefak sistem, bukan sinyal genuine.

In [7]:
train['discovered'] = pd.to_datetime(train['discovered'], format='%m/%d/%Y %H:%M')
train['registration_date'] = pd.to_datetime(train['registration_date'], format='%m/%d/%Y')
test['discovered'] = pd.to_datetime(test['discovered'], format='%m/%d/%Y %H:%M')
test['registration_date'] = pd.to_datetime(test['registration_date'], format='%m/%d/%Y')

## 8. Feature: Domain Age

Selisih antara `discovered` dan `registration_date` — fitur **paling robust**, terbukti causal & divalidasi riset eksternal.

- `domain_age_days` → selisih dalam hari (median-imputed jika `NaN`)
- `domain_age_is_negative` → flag jika domain ditemukan *sebelum* terdaftar (sinyal anomali)

In [8]:
train['domain_age_days'] = (train['discovered'] - train['registration_date']).dt.days
test['domain_age_days'] = (test['discovered'] - test['registration_date']).dt.days

train['domain_age_is_negative'] = (train['domain_age_days'] < 0).astype(int)
test['domain_age_is_negative'] = (test['domain_age_days'] < 0).astype(int)

train['domain_age_days'] = train['domain_age_days'].fillna(train['domain_age_days'].median())
test['domain_age_days'] = test['domain_age_days'].fillna(train['domain_age_days'].median())

## 9. Feature: URL Length

> `url_has_base64` di-drop — kontribusinya kecil & gap antar kelas tipis.

In [9]:
train['url_length'] = train['url'].str.len()
test['url_length'] = test['url'].str.len()

## 10. Feature: Brand Missing Flag

Flag `brand_is_missing = True` jika kolom brand kosong (`NaN`) atau berisi `'-'`.

In [10]:
for df in [train, test]:
    df['brand_is_missing'] = df['brand'].isna() | (df['brand'].str.strip() == '-')

## 11. Feature: One-Hot Encoding SLD

SLD (Second-Level Domain) di-encode karena kategorinya **terbatas & stabil** — tidak akan "kadaluarsa".

Kolom yang ada di train tapi tidak di test (atau sebaliknya) diisi `0`.

In [11]:
train = pd.get_dummies(train, columns=['sld'], prefix='sld')
test = pd.get_dummies(test, columns=['sld'], prefix='sld')

train_sld_cols = set(c for c in train.columns if c.startswith('sld_'))
test_sld_cols = set(c for c in test.columns if c.startswith('sld_'))

for col in train_sld_cols - test_sld_cols:
    test[col] = 0
for col in test_sld_cols - train_sld_cols:
    train[col] = 0

## 12. Sanity Check & Simpan Hasil Cleaning

In [12]:
sld_cols = [c for c in train.columns if c.startswith('sld_')]
feature_cols = [
    'domain_age_days', 'domain_age_is_negative',
    'url_length', 'brand_is_missing',
] + sld_cols

print("Missing value tersisa (train):")
print(train[feature_cols].isna().sum()[train[feature_cols].isna().sum() > 0])
print("\nMissing value tersisa (test):")
print(test[feature_cols].isna().sum()[test[feature_cols].isna().sum() > 0])

os.makedirs('../../data/result/v4', exist_ok=True)
train.to_csv('../../data/result/v4/train_final.csv', index=False)
test.to_csv('../../data/result/v4/test_final.csv', index=False)
print("\nTersimpan!")

Missing value tersisa (train):
Series([], dtype: int64)

Missing value tersisa (test):
Series([], dtype: int64)

Tersimpan!


## 13. Baseline Model (Eksperimen Internal)

Split train/validation (80/20 stratified) lalu latih **Random Forest** sebagai baseline.

> ⚠️ Cell ini **tidak menyentuh** `test.csv` / `predict.csv` — murni untuk evaluasi internal.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = train[feature_cols]
y = train['category']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

print(classification_report(y_val, model.predict(X_val), zero_division=0))

                 precision    recall  f1-score   support

          brand       0.67      0.44      0.53         9
       fakeshop       0.00      0.00      0.00         1
        malware       0.58      0.42      0.49        33
online gambling       0.96      0.98      0.97      1099
          other       0.89      0.89      0.89        56
       phishing       0.97      0.94      0.95       448
           spam       0.70      0.76      0.73        34

       accuracy                           0.95      1680
      macro avg       0.68      0.64      0.65      1680
   weighted avg       0.94      0.95      0.95      1680



## 14. Feature Importance

Validasi ulang fitur yang paling berpengaruh — top 15 ditampilkan.

In [14]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance.head(15))

domain_age_days           0.326021
url_length                0.322534
brand_is_missing          0.095349
sld_go.id                 0.063310
sld_id                    0.049043
domain_age_is_negative    0.034382
sld_sch.id                0.023167
sld_co.id                 0.022588
sld_my.id                 0.022489
sld_ac.id                 0.017896
sld_biz.id                0.007352
sld_net.id                0.006682
sld_or.id                 0.004221
sld_ponpes.id             0.002305
sld_desa.id               0.001526
dtype: float64


## 15. Retrain Full & Prediksi

Model diretrain menggunakan **seluruh data train** (tanpa validation split), lalu digunakan untuk prediksi ke data test.

Hasil disimpan ke `data/result/v4/submission.csv`.

In [15]:
final_model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
final_model.fit(X, y)

X_test = test[feature_cols]
predictions = final_model.predict(X_test)

submission = pd.read_csv('../../data/raw/submission-template.csv')
submission['category'] = predictions
submission.to_csv('../../data/result/v4/submission.csv', index=False)
print("Submission tersimpan!")
submission.head()

Submission tersimpan!


,id,category
0,PEDAS-bf8bc60352a3,phishing
1,PEDAS-7dd13e6859b0,online gambling
2,PEDAS-0fe0d926e0d2,phishing
3,PEDAS-37db191c480d,phishing
4,PEDAS-2d06b168bfcc,online gambling
